In [1]:
import numpy as np

def generate_shell_dataset_random_center(
    n_samples=200,
    image_size=128,
    radius_min=20,
    radius_max=50,
    ratio_min=0.4,
    ratio_max=0.8,
    center_std=5.0,  # 新增：控制中心点偏移程度的标准差
    dtype=np.float32
):
    """
    Generate circular shell dataset with anti-aliasing and randomized centers.
    
    Parameters:
    -----------
    center_std : float
        The standard deviation of the Gaussian distribution for the center coordinates.
        Larger values mean the circle will drift further from the exact center.
    """

    dataset = np.zeros(
        (n_samples, 1, image_size, image_size),
        dtype=dtype
    )

    # 坐标网格 (x, y) 只需要生成一次
    y, x = np.indices((image_size, image_size))
    
    # 图像的正中心基准点
    base_center = (image_size - 1) / 2.0

    for i in range(n_samples):
        # 1. 生成随机中心点 (高斯分布)
        # loc 是期望值(正中心)，scale 是标准差(偏移程度)
        cx = np.random.normal(loc=base_center, scale=center_std)
        cy = np.random.normal(loc=base_center, scale=center_std)

        # 2. 计算当前随机中心点对应的距离矩阵 r (必须在循环内计算)
        r = np.sqrt((x - cx)**2 + (y - cy)**2)

        # 3. 随机生成内外半径参数
        r_out = np.random.uniform(radius_min, radius_max)
        ratio = np.random.uniform(ratio_min, ratio_max)
        r_in = ratio * r_out

        # 4. 软边缘处理 (Anti-aliasing)
        outer_mask = np.clip(r_out - r + 0.5, 0.0, 1.0)
        inner_mask = np.clip(r - r_in + 0.5, 0.0, 1.0)

        # 5. 组合掩码并赋值
        shell = (outer_mask * inner_mask).astype(dtype)
        dataset[i, 0] = shell

    return dataset


# ==================================================
# Example
# ==================================================

dataset = generate_shell_dataset_random_center(
    n_samples=200,
    image_size=128,
    radius_min=10,
    radius_max=30,
    ratio_min=0.5,
    ratio_max=0.8,
    center_std=10.0  # 圆环半径较小，标准差也设置小一点，防止圆环跑到图像外部
)

print("Dataset shape:", dataset.shape)

# save
np.save("shell_dataset_add.npy", dataset)

Dataset shape: (200, 1, 128, 128)


In [2]:
import numpy as np

def generate_elliptical_shell_dataset(
    n_samples=200,
    image_size=128,
    a_min=20, a_max=50,      # 椭圆半轴 A 的随机范围
    b_min=10, b_max=30,      # 椭圆半轴 B 的随机范围
    ratio_min=0.4, ratio_max=0.8,
    center_std=5.0,          # 中心点偏移标准差
    dtype=np.float32
):
    """
    Generate an elliptical shell dataset with anti-aliasing, randomized centers, and random rotations.
    """
    dataset = np.zeros(
        (n_samples, 1, image_size, image_size),
        dtype=dtype
    )

    # 坐标网格只需要生成一次
    y, x = np.indices((image_size, image_size))
    base_center = (image_size - 1) / 2.0

    for i in range(n_samples):
        # 1. 随机中心点
        cx = np.random.normal(loc=base_center, scale=center_std)
        cy = np.random.normal(loc=base_center, scale=center_std)

        # 2. 随机椭圆参数 (长半轴、短半轴、旋转角度)
        a = np.random.uniform(a_min, a_max)
        b = np.random.uniform(b_min, b_max)
        theta = np.random.uniform(0, np.pi)  # 随机旋转 0 到 180 度

        # 3. 随机内外环比例
        ratio = np.random.uniform(ratio_min, ratio_max)

        # 4. 坐标平移到随机中心
        dx = x - cx
        dy = y - cy

        # 5. 坐标系旋转 (抵消椭圆的旋转)
        x_rot = dx * np.cos(theta) + dy * np.sin(theta)
        y_rot = -dx * np.sin(theta) + dy * np.cos(theta)

        # 6. 计算每个像素到中心的距离 (极径 r) 和 角度 (极角 phi)
        r_pixel = np.sqrt(x_rot**2 + y_rot**2)
        # 避免除以零的警告，当 r_pixel 为 0 时 arctan2 的结果是 0，不影响后续计算
        phi = np.arctan2(y_rot, x_rot) 

        # 7. 根据极坐标方程计算在每个角度 phi 下，椭圆外边界的精确半径
        r_out_boundary = (a * b) / np.sqrt((b * np.cos(phi))**2 + (a * np.sin(phi))**2)
        
        # 内边界的精确半径
        r_in_boundary = r_out_boundary * ratio

        # 8. 软边缘处理 (与圆环逻辑完全一致，因为我们已经把椭圆转化为了每个角度的半径比对)
        outer_mask = np.clip(r_out_boundary - r_pixel + 0.5, 0.0, 1.0)
        inner_mask = np.clip(r_pixel - r_in_boundary + 0.5, 0.0, 1.0)

        # 9. 组合掩码
        shell = (outer_mask * inner_mask).astype(dtype)
        dataset[i, 0] = shell

    return dataset


# ==================================================
# Example
# ==================================================

dataset = generate_elliptical_shell_dataset(
    n_samples=200,
    image_size=128,
    a_min=12, a_max=36,   # 半轴 A
    b_min=6, b_max=12,    # 半轴 B (B 明显小于 A，确保形状是明显的椭圆)
    ratio_min=0.4,
    ratio_max=0.8,
    center_std=10.0        # 中心偏移
)

print("Dataset shape:", dataset.shape)

# save
np.save("elliptical_shell_dataset_add.npy", dataset)

Dataset shape: (200, 1, 128, 128)
